# RQ2 

## Setup

* Follow the instructions on ```Readme.md``` 
* Inside the *pipeline* folder, run ```python -m src.rq2```. This will parse the .json file containing the dataset into  ```bigcodebench_clone_dataset.xml``` file containing all clone pairs (exclusing the original code). 
* Checkout [CloneCognition](https://github.com/pseudoPixels/CloneCognition)
* Paste the ```bigcodebench_clone_dataset.xml``` file inside ```input_clone_pairs/``` on the CloneCognition project.
    * Make sure to follow the CloneCognition instructions on their README
* Run ```python validateClones.py 0.76 input_clone_pairs/ out/```. This will produce a file  ```bigcodebench_clone_dataset.xml.mlValidated```
* The file ```bigcodebench_clone_dataset.xml.mlValidated``` is available in our repo under ```results/RQ2```

In [2]:
from src.config import *
PAIRS = f"../results/RQ2/{DATASET_NAME}_clone_dataset.xml"
RESULTS = f"../results/RQ2/{DATASET_NAME}_clone_dataset.xml.mlValidated"

In [3]:
with open(PAIRS, "r", encoding="utf-8") as f:
    xml_text = f.read()

num_clones = xml_text.count("</clone>")
print("Number of clone pairs:", num_clones)

Number of clone pairs: 56362


## Results

In [4]:
# Summary
from collections import Counter
 
counts = Counter()
total = 0

with open(RESULTS, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue  # skip empty lines

        first_field = line.split(",", 1)[0].lower()
        if first_field in {"true", "false"}:
            counts[first_field] += 1
            total += 1

true_count = counts.get("true", 0)
false_count = counts.get("false", 0)

true_pct = (true_count / total * 100) if total > 0 else 0
false_pct = (false_count / total * 100) if total > 0 else 0

print(f"Total entries : {total}")
print(f"Detected clones : {true_count} ({true_pct:.2f}%)")
print(f"Not detected : {false_count} ({false_pct:.2f}%)")


Total entries : 56362
Detected clones : 51 (0.09%)
Not detected : 56311 (99.91%)


In [5]:
# Extract and print entries marked as true
true_entries = []

with open(RESULTS, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue

        cols = [c.strip() for c in line.split(",")]

        if cols[0].lower() == "true" and len(cols) >= 4:
            true_entries.append(((cols[1], cols[2], cols[3]), (cols[6], cols[7], cols[8])))
 
print("Detected clones:")
for id1, id2 in true_entries:
    print(f"{id1}, {id2}")

print(f"\nTotal detected clones: {len(true_entries)}")

Detected clones:
("BigCodeBench/676_cot deepseek-r1:14b-test 1 ['refac_2'", "'refac_6'", "'refac_7']"), ("BigCodeBench/676_cot deepseek-r1:14b-complete 1 ['refac_1'", "'refac_3'", "'refac_7']")
("BigCodeBench/293_zero-shot deepseek-r1:14b-test 1 ['refac_1'", "'refac_3'", "'refac_4']"), ("BigCodeBench/293_zero-shot deepseek-r1:14b-test 1 ['refac_2'", "'refac_6'", "'refac_7']")
("BigCodeBench/33_zero-shot gpt-oss:20b-ast 1 ['refac_1'", "'refac_3'", "'refac_4']"), ("BigCodeBench/33_cot gpt-oss:20b-ast 1 ['refac_1'", "'refac_3'", "'refac_4']")
("BigCodeBench/33_cot gemma3:latest-test 1 ['refac_2'", "'refac_6'", "'refac_7']"), ("BigCodeBench/33_cot llama3.1:latest-test 1 ['refac_1'", "'refac_3'", "'refac_7']")
("BigCodeBench/697_zero-shot gemma3:latest-complete 1 ['refac_2'", "'refac_6'", "'refac_7']"), ("BigCodeBench/697_zero-shot deepseek-r1:14b-complete 1 ['refac_1'", "'refac_4'", "'refac_5']")
("BigCodeBench/275_zero-shot llama3.1:latest-code 1 ['refac_2'", "'refac_4'", "'refac_6']"), (

In [11]:
import pandas as pd
import re
from collections import Counter


with open(RESULTS, "r") as f:
    lines = [l.strip() for l in f if l.strip()]

def parse_side(text):
    entry = re.search(r"(BigCodeBench/\d+)", text)
    strategy = re.search(r"_(cot|zero-shot)", text)
    model = re.search(r"(llama3\.1|deepseek-r1|gpt-oss|gemma3)", text)

    # >>> ADDED: context extraction
    context = re.search(r"-(test|ast|code|complete)\b", text)

    refacs = re.findall(r"refac_\d+", text)

    return {
        "entry": entry.group(1) if entry else None,
        "strategy": strategy.group(1) if strategy else None,
        "model": model.group(1) if model else None,
        "context": context.group(1) if context else None,   # <<< ADDED
        "refacs": refacs,
    }

records = []

for line in lines:
    parts = [p.strip() for p in line.split(",")]

    # Column 0 = TRUE / FALSE
    if parts[0].lower() != "true":
        continue

    # Left and right clone descriptions
    left_text = parts[1]
    right_text = parts[4]

    left = parse_side(left_text)
    right = parse_side(right_text)

    records.append({
        "entry": left["entry"],
        "strategy_left": left["strategy"],
        "strategy_right": right["strategy"],
        "model_left": left["model"],
        "model_right": right["model"],
        "context_left": left["context"],    # <<< ADDED
        "context_right": right["context"],  # <<< ADDED
        "refacs_left": left["refacs"],
        "refacs_right": right["refacs"],
    })

df = pd.DataFrame(records)

entry_counts = df["entry"].value_counts()

strategy_counts = Counter(df["strategy_left"]) + Counter(df["strategy_right"])
model_counts = Counter(df["model_left"]) + Counter(df["model_right"])

# >>> ADDED: context counter
context_counts = Counter(df["context_left"]) + Counter(df["context_right"])

refac_counts = Counter()
for _, r in df.iterrows():
    refac_counts.update(r["refacs_left"])
    refac_counts.update(r["refacs_right"])

strategy_pairs = Counter(
    zip(df["strategy_left"], df["strategy_right"])
)

print("\n=== Most problematic BigCodeBench entries ===")
print(entry_counts.head(10))

print("\n=== Strategy involvement ===")
for k, v in strategy_counts.most_common():
    if k is None:
        continue
    print(f"{k:10s} {v}")

print("\n=== Model involvement ===")
for k, v in model_counts.most_common():
    if k is None:
        continue
    print(f"{k:15s} {v}")

# >>> ADDED: context output
print("\n=== Context involvement ===")
for k, v in context_counts.most_common():
    if k is None:
        continue
    print(f"{k:10s} {v}")

print("\n=== Refactorings involved ===")
for k, v in refac_counts.most_common():
    print(f"{k:8s} {v}")

print("\n=== Strategy pairings ===")
for (s1, s2), v in strategy_pairs.most_common():
    print(f"{s1} vs {s2}: {v}")



=== Most problematic BigCodeBench entries ===
entry
BigCodeBench/911    10
BigCodeBench/546     2
BigCodeBench/122     2
BigCodeBench/4       2
BigCodeBench/33      2
BigCodeBench/667     2
BigCodeBench/697     1
BigCodeBench/293     1
BigCodeBench/676     1
BigCodeBench/275     1
Name: count, dtype: int64

=== Strategy involvement ===
zero-shot  33
cot        18

=== Model involvement ===
deepseek-r1     19
gemma3          15
gpt-oss         10
llama3.1        7

=== Context involvement ===
test       21
ast        11
complete   10
code       9

=== Refactorings involved ===
refac_1  39
refac_2  11
refac_3  1

=== Strategy pairings ===
zero-shot vs None: 33
cot vs None: 18
